# Background Jobs and Batch APIs

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/09-background-jobs-batch-apis.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi httpx pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 09 - Background Jobs and Batch APIs
Phase 06: Shipping

Demonstrates two async LLM patterns:
1. In-memory job queue with FastAPI BackgroundTasks (POST job, poll GET /jobs/{id})
2. Anthropic Batch API for bulk requests at 50% cost discount

Run:
    uv pip install fastapi uvicorn anthropic httpx
    ANTHROPIC_API_KEY=sk-... uvicorn main:app --reload
"""
import os
import uuid
from datetime import datetime
from enum import Enum
from typing import Any

import anthropic
from fastapi import BackgroundTasks, FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Background Jobs Demo", version="1.0.0")
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# In-memory store.
# In production, replace with Redis (job queue) or Postgres (job records).
jobs: dict[str, dict[str, Any]] = {}

### Data models

In [ ]:
class JobStatus(str, Enum):
    pending = "pending"
    running = "running"
    done = "done"
    failed = "failed"


class GenerateRequest(BaseModel):
    text: str
    instruction: str = "Summarize the following in one paragraph."


class JobResponse(BaseModel):
    job_id: str
    status: JobStatus
    result: str | None = None
    error: str | None = None
    created_at: str
    completed_at: str | None = None

### Background worker

In [ ]:
def run_generation(job_id: str, text: str, instruction: str) -> None:
    """
    Blocking Anthropic call. FastAPI BackgroundTasks runs this in a thread pool
    so it does not block the event loop.
    Updates the jobs store when complete.
    """
    jobs[job_id]["status"] = JobStatus.running
    try:
        message = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=512,
            messages=[
                {
                    "role": "user",
                    "content": f"{instruction}\n\n{text}",
                }
            ],
        )
        jobs[job_id]["status"] = JobStatus.done
        jobs[job_id]["result"] = message.content[0].text
        jobs[job_id]["completed_at"] = datetime.utcnow().isoformat()
    except Exception as exc:
        jobs[job_id]["status"] = JobStatus.failed
        jobs[job_id]["error"] = str(exc)
        jobs[job_id]["completed_at"] = datetime.utcnow().isoformat()

### Job queue endpoints

In [ ]:
@app.post("/jobs", response_model=JobResponse, status_code=202)
async def create_job(
    request: GenerateRequest,
    background_tasks: BackgroundTasks,
) -> JobResponse:
    """
    Accept work, return job_id immediately (HTTP 202), start worker in background.
    The client must poll GET /jobs/{job_id} to retrieve the result.
    """
    job_id = str(uuid.uuid4())[:8]
    jobs[job_id] = {
        "job_id": job_id,
        "status": JobStatus.pending,
        "result": None,
        "error": None,
        "created_at": datetime.utcnow().isoformat(),
        "completed_at": None,
    }
    background_tasks.add_task(run_generation, job_id, request.text, request.instruction)
    return JobResponse(**jobs[job_id])


@app.get("/jobs/{job_id}", response_model=JobResponse)
async def get_job(job_id: str) -> JobResponse:
    """
    Poll this endpoint to check job status and retrieve the result.

    Status transitions: pending -> running -> done | failed
    Poll every 2-5 seconds. Stop when status is 'done' or 'failed'.
    """
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail=f"Job '{job_id}' not found")
    return JobResponse(**jobs[job_id])


@app.get("/jobs", response_model=list[JobResponse])
async def list_jobs() -> list[JobResponse]:
    """List all jobs. Useful for debugging; paginate in production."""
    return [JobResponse(**job) for job in jobs.values()]

### Anthropic Batch API endpoints

In [ ]:
@app.post("/batch", status_code=202)
async def create_batch(texts: list[str]) -> dict[str, Any]:
    """
    Submit up to 10,000 texts to the Anthropic Batch API.
    Returns a batch_id. Results arrive within 24 hours at 50% cost discount.

    Request body: ["text1", "text2", ...]
    """
    if not texts:
        raise HTTPException(status_code=400, detail="texts list must not be empty")
    if len(texts) > 10_000:
        raise HTTPException(status_code=400, detail="Batch API limit is 10,000 requests")

    requests_list = [
        {
            "custom_id": f"item-{i}",
            "params": {
                "model": "claude-3-5-haiku-20241022",
                "max_tokens": 256,
                "messages": [
                    {
                        "role": "user",
                        "content": f"Summarize in one sentence: {text}",
                    }
                ],
            },
        }
        for i, text in enumerate(texts)
    ]

    batch = client.messages.batches.create(requests=requests_list)
    return {
        "batch_id": batch.id,
        "status": batch.processing_status,
        "request_count": len(texts),
    }


@app.get("/batch/{batch_id}")
async def get_batch_status(batch_id: str) -> dict[str, Any]:
    """
    Poll for batch completion. When status is 'ended', results are included.

    processing_status values: 'in_progress' | 'canceling' | 'ended'
    Poll every 30-60 seconds. Batches typically complete within a few minutes
    for small batches, up to 24 hours for very large ones.
    """
    batch = client.messages.batches.retrieve(batch_id)

    if batch.processing_status != "ended":
        return {
            "batch_id": batch_id,
            "status": batch.processing_status,
            "request_counts": batch.request_counts.model_dump(),
        }

    # Batch ended. Stream results and collect.
    results: dict[str, str] = {}
    errors: dict[str, str] = {}

    for result in client.messages.batches.results(batch_id):
        if result.result.type == "succeeded":
            results[result.custom_id] = result.result.message.content[0].text
        else:
            errors[result.custom_id] = (
                result.result.error.type
                if hasattr(result.result, "error")
                else "unknown_error"
            )

    return {
        "batch_id": batch_id,
        "status": "ended",
        "succeeded": len(results),
        "errored": len(errors),
        "results": results,
        "errors": errors,
    }


# ---------------------------------------------------------------------------
# Demo client (run as a script to test without a real server)
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import time

import httpx

BASE = "http://localhost:8000"

print("=== Demo: Job Pattern ===")
resp = httpx.post(f"{BASE}/jobs", json={"text": "FastAPI is a modern Python web framework."})
print(f"POST /jobs -> {resp.status_code}")
data = resp.json()
job_id = data["job_id"]
print(f"Job ID: {job_id}, Status: {data['status']}")

for attempt in range(15):
    time.sleep(3)
    r = httpx.get(f"{BASE}/jobs/{job_id}")
    status = r.json()["status"]
    print(f"  Poll {attempt + 1}: {status}")
    if status in ("done", "failed"):
        print(f"  Result: {r.json().get('result') or r.json().get('error')}")
        break

print("\n=== Demo: Batch API ===")
texts = [
    "Python is a high-level programming language.",
    "FastAPI is built on Starlette and Pydantic.",
    "Anthropic builds AI safety research.",
]
resp = httpx.post(f"{BASE}/batch", json=texts)
print(f"POST /batch -> {resp.status_code}")
batch_data = resp.json()
batch_id = batch_data["batch_id"]
print(f"Batch ID: {batch_id}")

for attempt in range(10):
    time.sleep(30)
    r = httpx.get(f"{BASE}/batch/{batch_id}")
    data = r.json()
    print(f"  Poll {attempt + 1}: {data['status']}")
    if data["status"] == "ended":
        for cid, result in data["results"].items():
            print(f"  {cid}: {result[:80]}...")
        break

---

## Self-Check

Answer these without running code first:

**1. What architecture is correct for this feature?**

- A. Loop over 500 documents in the POST handler and return all results when done.
- B. Accept the request, enqueue the work, return a job_id, and let the client poll for completion.
- C. Use streaming SSE so the user sees each summary appear as it is generated.
- D. Increase the server timeout to 70 minutes so the synchronous loop can complete.

**2. Who is correct and why?**

- A. Your colleague. HTTP 200 means success, and creating the job was successful.
- B. You. HTTP 202 means 'accepted for processing' and signals the work is not done yet.
- C. Neither. Async endpoints should return HTTP 201 Created.
- D. It does not matter. HTTP status codes are informational only and clients ignore them.

**3. Which Anthropic API surface should you use?**

- A. Standard Messages API with asyncio to parallelize the 2,000 calls.
- B. Anthropic Batch API, which processes up to 10,000 requests within 24 hours at 50% cost.
- C. Streaming Messages API to reduce per-request latency.
- D. Standard Messages API with a retry loop, since batch processing adds overhead.

**4. What is the root cause and what is the correct fix?**

- A. BackgroundTasks does not start before the server shuts down. Fix by increasing the shutdown timeout.
- B. The in-memory jobs dict is lost on restart. Fix by persisting job records to Redis or Postgres.
- C. FastAPI cancels background tasks on restart by design. Fix by switching to Celery.
- D. The jobs dict is not thread-safe. Fix by wrapping it in a threading.Lock.

**5. What is the correct handling for the errored items?**

- A. Discard them. Errors in batch processing are expected and acceptable.
- B. Re-submit only the errored custom_ids as a new batch, then merge results.
- C. Fail the entire batch job and re-submit all 50 items.
- D. Switch to the standard API for all future batches since errors indicate the Batch API is unreliable.

**6. What problem does this create and how do you fix it?**

- A. No problem. Polling is the correct pattern and 500ms is a reasonable interval.
- B. 1,000 users polling every 500ms = 2,000 GET requests per second hitting the job store. Use exponential backoff or longer intervals (2-5 seconds) to reduce load.
- C. Polling creates a race condition where two clients can claim the same job result.
- D. The job store dict cannot handle more than 100 concurrent readers. Switch to a queue.

**7. What is the correct approach?**

- A. Submit all 15,000 in one batch. The API will silently drop the last 5,000.
- B. Split into two batches (10,000 and 5,000), submit both, and merge results when both are ended.
- C. Use the standard API for the 5,000 overflow items and the Batch API for the first 10,000.
- D. Upgrade to Anthropic Enterprise tier to unlock a higher batch limit.

**8. Which tool is the better choice for this workload?**

- A. FastAPI BackgroundTasks. It is simpler and 50-100 jobs per hour is a low volume.
- B. Celery with a Redis broker. The durability requirement rules out BackgroundTasks.
- C. Neither. Use asyncio.gather to run all jobs concurrently in a single request.
- D. BackgroundTasks with a file-based job store to survive restarts.

_Answers are in `checks.json` in the lesson directory._